# 手搓 LangGraph 最小形态
如果将所有的：调用 LLM、Tool_call 都写在同一个 Agent_run （Agent Runtime）里面，那么这个函数会越来越膨胀变大！所以，LangGraph 本质上是一个重构，将不同的操作抽象为一个个 Node：
- llm_node 负责调用 LLM，并返回决策结果
- tool_node 负责调用工具，返回处理结果
- runtime 负责整个流程的执行

```shell
             ┌─────────────┐
             │ LLM Node    │
             └──────┬──────┘
                    │
             Tool Call?
               /       \
             YES        NO
              ↓          ↓
       ┌─────────────┐   END
       │ Tool Node   │
       └──────┬──────┘
              │
              ↓
          LLM Node
```

## node 定义示例

In [ ]:
import json

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field


# ============================================================
# 1. Load environment variables
# ============================================================

load_dotenv()


# ============================================================
# 2. Create LLM client
# ============================================================

client = OpenAI()


# ============================================================
# 3. Agent State
# ============================================================

class AgentState(BaseModel):
    """
    保存 Agent 在整个任务执行过程中的状态。
    """

    # 用户最初提出的任务
    task: str

    # 当前执行轮数
    iteration: int = 0

    # Agent 是否完成
    finished: bool = False

    # LLM 请求过的工具
    tool_calls: list = Field(
        default_factory=list
    )

    # 工具执行结果
    tool_results: list = Field(
        default_factory=list
    )


# ============================================================
# 4. Tools
# ============================================================

def calculator(
    a: float,
    b: float,
    operation: str,
) -> float:
    """
    执行基本数学计算。
    """

    if operation == "add":
        return a + b

    if operation == "subtract":
        return a - b

    if operation == "multiply":
        return a * b

    if operation == "divide":

        if b == 0:
            raise ValueError(
                "Cannot divide by zero."
            )

        return a / b

    raise ValueError(
        f"Unknown operation: {operation}"
    )


def get_user_name() -> str:
    """
    获取当前用户的名字。
    """

    return "Sylvie"


# ============================================================
# 5. Tool Registry
# ============================================================

tool_registry = {
    "calculator": calculator,
    "get_user_name": get_user_name,
}


# ============================================================
# 6. Tool Schemas
# ============================================================

tools = [
    {
        "type": "function",
        "name": "calculator",
        "description": "执行基本数学计算。",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "number",
                    "description": "第一个数字。",
                },
                "b": {
                    "type": "number",
                    "description": "第二个数字。",
                },
                "operation": {
                    "type": "string",
                    "enum": [
                        "add",
                        "subtract",
                        "multiply",
                        "divide",
                    ],
                    "description": "计算操作。",
                },
            },
            "required": [
                "a",
                "b",
                "operation",
            ],
        },
    },
    {
        "type": "function",
        "name": "get_user_name",
        "description": "获取当前用户的名字。",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
        },
    },
]


# ============================================================
# 7. LLM Node
# ============================================================

def llm_node(
    input_items,
    previous_response_id=None,
):
    """
    LLM Node：

    负责调用 LLM，
    让 LLM 决定下一步应该做什么。
    """

    response = client.responses.create(
        model="gpt-5",
        input=input_items,
        tools=tools,
        previous_response_id=previous_response_id,
    )

    return response


# ============================================================
# 8. Tool Node
# ============================================================

def tool_node(
    response,
    state: AgentState,
):
    """
    Tool Node：

    负责执行 LLM 请求的所有工具。

    返回：
        tool_outputs

    这些结果之后会重新发送给 LLM。
    """

    tool_outputs = []

    # --------------------------------------------------------
    # 遍历 LLM 当前 Response 中的所有输出
    # --------------------------------------------------------

    for item in response.output:

        # ----------------------------------------------------
        # 只处理 function_call
        # ----------------------------------------------------

        if item.type != "function_call":
            continue

        print(
            f"Tool Call: {item.name}"
        )

        print(
            f"Arguments: {item.arguments}"
        )

        # ----------------------------------------------------
        # 1. 记录 Tool Call
        # ----------------------------------------------------

        state.tool_calls.append(
            {
                "name": item.name,
                "arguments": item.arguments,
                "call_id": item.call_id,
            }
        )

        # ----------------------------------------------------
        # 2. 根据 Tool Name 找到 Python 函数
        # ----------------------------------------------------

        tool_name = item.name

        if tool_name not in tool_registry:
            raise ValueError(
                f"Unknown tool: {tool_name}"
            )

        tool = tool_registry[tool_name]

        # ----------------------------------------------------
        # 3. 解析参数
        # ----------------------------------------------------

        arguments = json.loads(
            item.arguments
        )

        # ----------------------------------------------------
        # 4. 执行 Tool
        # ----------------------------------------------------

        result = tool(
            **arguments
        )

        print(
            f"Tool Result: {result}"
        )

        # ----------------------------------------------------
        # 5. 记录 Tool Result
        # ----------------------------------------------------

        state.tool_results.append(
            {
                "call_id": item.call_id,
                "tool": tool_name,
                "result": result,
            }
        )

        # ----------------------------------------------------
        # 6. 构造 function_call_output
        # ----------------------------------------------------

        tool_outputs.append(
            {
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": str(result),
            }
        )

    return tool_outputs


# ============================================================
# 9. Main Agent Runtime
# ============================================================

def run_agent(task: str):

    # --------------------------------------------------------
    # Create State
    # --------------------------------------------------------

    state = AgentState(
        task=task
    )

    # --------------------------------------------------------
    # Initial LLM Input
    # --------------------------------------------------------

    input_items = [
        {
            "role": "user",
            "content": task,
        }
    ]

    # --------------------------------------------------------
    # First LLM Node
    # --------------------------------------------------------

    response = llm_node(
        input_items=input_items
    )

    # ========================================================
    # Agent Loop
    # ========================================================

    while not state.finished:

        # ----------------------------------------------------
        # Update iteration
        # ----------------------------------------------------

        state.iteration += 1

        print(
            f"\n--- Agent Iteration "
            f"{state.iteration} ---"
        )

        # ----------------------------------------------------
        # Safety limit
        # ----------------------------------------------------

        if state.iteration > 10:

            raise RuntimeError(
                "Agent exceeded maximum iterations."
            )

        # ----------------------------------------------------
        # Tool Node
        # ----------------------------------------------------

        tool_outputs = tool_node(
            response=response,
            state=state,
        )

        # ----------------------------------------------------
        # No Tool Call
        # ----------------------------------------------------

        if not tool_outputs:

            state.finished = True

            print(
                "\nFinal Answer:"
            )

            print(
                response.output_text
            )

            break

        # ----------------------------------------------------
        # Send Tool Results back to LLM
        # ----------------------------------------------------

        response = llm_node(
            input_items=tool_outputs,
            previous_response_id=response.id,
        )

    return state


# ============================================================
# 10. Run
# ============================================================

if __name__ == "__main__":

    task = (
        "请告诉我我的名字，然后计算 "
        "12345 + 6789，再把结果乘以 2。"
    )

    final_state = run_agent(task)

    # ========================================================
    # Print Final State
    # ========================================================

    print("\n" + "=" * 60)
    print("FINAL AGENT STATE")
    print("=" * 60)

    print(
        f"\nTask:\n{final_state.task}"
    )

    print(
        f"\nIterations:\n"
        f"{final_state.iteration}"
    )

    print(
        f"\nFinished:\n"
        f"{final_state.finished}"
    )

    print("\nTool Calls:")

    for call in final_state.tool_calls:
        print(call)

    print("\nTool Results:")

    for result in final_state.tool_results:
        print(result)


--- Agent Iteration 1 ---
Tool Call: get_user_name
Arguments: {}
Tool Result: Sylvie
Tool Call: calculator
Arguments: {"a":12345,"b":6789,"operation":"add"}
Tool Result: 19134

--- Agent Iteration 2 ---
Tool Call: calculator
Arguments: {"a":19134,"b":2,"operation":"multiply"}
Tool Result: 38268

--- Agent Iteration 3 ---

Final Answer:
你的名字是 Sylvie。
计算结果：12345 + 6789 = 19134，再乘以 2 得到 38268。

FINAL AGENT STATE

Task:
请告诉我我的名字，然后计算 12345 + 6789，再把结果乘以 2。

Iterations:
3

Finished:
True

Tool Calls:
{'name': 'get_user_name', 'arguments': '{}', 'call_id': 'call_V80PSjNaFYQIReyY11fbhqCN'}
{'name': 'calculator', 'arguments': '{"a":12345,"b":6789,"operation":"add"}', 'call_id': 'call_miizsOA8yb77DKnUoAnqZv0z'}
{'name': 'calculator', 'arguments': '{"a":19134,"b":2,"operation":"multiply"}', 'call_id': 'call_EcaZraWV45tVkFvvzHlBwHT0'}

Tool Results:
{'call_id': 'call_V80PSjNaFYQIReyY11fbhqCN', 'tool': 'get_user_name', 'result': 'Sylvie'}
{'call_id': 'call_miizsOA8yb77DKnUoAnqZv0z', 'tool': 'calc

## MiniGraph Runtime 的实现
引出 LangGraph 的几个概念：
- node：
    - 包含所有执行节点
    - 接收 State、修改/返回 State 的函数。
- edge
    - conditional edge：条件判断是否经过该边
- start_node

----
下面代码，进一步抽象 runtime，并不知道执行的细节，只知道要交给下个 node，传给它 state，然后把结果 state 再传给下个 node……一路执行 Graph

state 作为 node 之间传递信息的媒介！类似于 piania 状态管理，nodes 从 state 读取信息，处理后传给下一个 state

```shell
             State₀
                ↓
            LLM Node
                ↓
             State₁
                ↓
           Tool Node
                ↓
             State₂
                ↓
            LLM Node
                ↓
             State₃
                ↓
             Tool
                ↓
             State₄
                ↓
             LLM
                ↓
             State₅
                ↓
              END
```


### miniGraph 代码示例

In [ ]:
import json

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field


# ============================================================
# 1. Load environment variables
# ============================================================

load_dotenv()


# ============================================================
# 2. LLM Client
# ============================================================

client = OpenAI()


# ============================================================
# 3. Agent State
# ============================================================

class AgentState(BaseModel):
    """
    Agent 在整个执行过程中的统一状态。

    所有 Node 都通过 State 进行通信。
    """

    # --------------------------------------------------------
    # Original user task
    # --------------------------------------------------------

    task: str

    # --------------------------------------------------------
    # Runtime state
    # --------------------------------------------------------

    iteration: int = 0

    finished: bool = False

    # --------------------------------------------------------
    # Tool history
    # --------------------------------------------------------

    tool_calls: list = Field(
        default_factory=list
    )

    tool_results: list = Field(
        default_factory=list
    )

    # --------------------------------------------------------
    # LLM state
    # --------------------------------------------------------

    last_response_id: str | None = None

    # --------------------------------------------------------
    # Pending actions
    # --------------------------------------------------------

    pending_tool_calls: list = Field(
        default_factory=list
    )

    pending_tool_outputs: list = Field(
        default_factory=list
    )

    # --------------------------------------------------------
    # Final answer
    # --------------------------------------------------------

    final_answer: str | None = None


# ============================================================
# 4. Tools
# ============================================================

def calculator(
    a: float,
    b: float,
    operation: str,
) -> float:

    if operation == "add":
        return a + b

    if operation == "subtract":
        return a - b

    if operation == "multiply":
        return a * b

    if operation == "divide":

        if b == 0:
            raise ValueError(
                "Cannot divide by zero."
            )

        return a / b

    raise ValueError(
        f"Unknown operation: {operation}"
    )


def get_user_name() -> str:

    return "Sylvie"


# ============================================================
# 5. Tool Registry
# ============================================================

tool_registry = {

    "calculator": calculator,

    "get_user_name": get_user_name,

}


# ============================================================
# 6. Tool Schemas
# ============================================================

tools = [

    {
        "type": "function",

        "name": "calculator",

        "description": "执行基本数学计算。",

        "parameters": {

            "type": "object",

            "properties": {

                "a": {
                    "type": "number",
                    "description": "第一个数字。",
                },

                "b": {
                    "type": "number",
                    "description": "第二个数字。",
                },

                "operation": {

                    "type": "string",

                    "enum": [
                        "add",
                        "subtract",
                        "multiply",
                        "divide",
                    ],

                    "description": "计算操作。",
                },

            },

            "required": [
                "a",
                "b",
                "operation",
            ],
        },
    },

    {
        "type": "function",

        "name": "get_user_name",

        "description": "获取当前用户的名字。",

        "parameters": {

            "type": "object",

            "properties": {},

            "required": [],
        },
    },

]


# ============================================================
# 7. LLM Node
# ============================================================

def llm_node(
    state: AgentState,
) -> AgentState:
    """
    LLM Node

    接口：

        State → State

    职责：

        1. 根据当前 State 调用 LLM
        2. 获取 LLM 决策
        3. 将 Tool Call / Final Answer 写入 State
    """

    # --------------------------------------------------------
    # Build LLM input
    # --------------------------------------------------------

    if state.last_response_id is None:

        input_items = [
            {
                "role": "user",
                "content": state.task,
            }
        ]

        response = client.responses.create(

            model="gpt-5",

            input=input_items,

            tools=tools,
        )

    else:

        input_items = state.pending_tool_outputs

        response = client.responses.create(

            model="gpt-5",

            previous_response_id=(
                state.last_response_id
            ),

            input=input_items,

            tools=tools,
        )

    # --------------------------------------------------------
    # Save Response ID
    # --------------------------------------------------------

    state.last_response_id = response.id

    # --------------------------------------------------------
    # Clear old pending calls
    # --------------------------------------------------------

    state.pending_tool_calls = []

    # --------------------------------------------------------
    # Inspect LLM response
    # --------------------------------------------------------

    for item in response.output:

        if item.type == "function_call":

            state.pending_tool_calls.append(

                {
                    "name": item.name,

                    "arguments": item.arguments,

                    "call_id": item.call_id,
                }
            )

    # --------------------------------------------------------
    # No Tool Call
    # --------------------------------------------------------

    if not state.pending_tool_calls:

        state.finished = True

        state.final_answer = (
            response.output_text
        )

    return state


# ============================================================
# 8. Tool Node
# ============================================================

def tool_node(
    state: AgentState,
) -> AgentState:
    """
    Tool Node

    接口：

        State → State

    职责：

        1. 读取 pending_tool_calls
        2. 执行工具
        3. 记录结果
        4. 生成 pending_tool_outputs
    """

    # --------------------------------------------------------
    # Clear previous outputs
    # --------------------------------------------------------

    state.pending_tool_outputs = []

    # --------------------------------------------------------
    # Execute every pending Tool Call
    # --------------------------------------------------------

    for call in state.pending_tool_calls:

        tool_name = call["name"]

        arguments = json.loads(
            call["arguments"]
        )

        call_id = call["call_id"]

        print(
            f"Tool Call: {tool_name}"
        )

        print(
            f"Arguments: {arguments}"
        )

        # ----------------------------------------------------
        # Find tool
        # ----------------------------------------------------

        if tool_name not in tool_registry:

            raise ValueError(
                f"Unknown tool: {tool_name}"
            )

        tool = tool_registry[
            tool_name
        ]

        # ----------------------------------------------------
        # Execute
        # ----------------------------------------------------

        result = tool(
            **arguments
        )

        print(
            f"Tool Result: {result}"
        )

        # ----------------------------------------------------
        # Record Tool Call
        # ----------------------------------------------------

        state.tool_calls.append(

            {
                "name": tool_name,

                "arguments": call["arguments"],

                "call_id": call_id,
            }
        )

        # ----------------------------------------------------
        # Record Tool Result
        # ----------------------------------------------------

        state.tool_results.append(

            {
                "call_id": call_id,

                "tool": tool_name,

                "result": result,
            }
        )

        # ----------------------------------------------------
        # Prepare LLM input
        # ----------------------------------------------------

        state.pending_tool_outputs.append(

            {
                "type": "function_call_output",

                "call_id": call_id,

                "output": str(result),
            }
        )

    return state


# ============================================================
# 9. Router
# ============================================================

def route_after_llm(
    state: AgentState,
) -> str:
    """
    根据 State 决定下一步。

    finished:
        → end

    有 pending tool calls:
        → tool

    否则:
        → end
    """

    if state.finished:

        return "end"

    if state.pending_tool_calls:

        return "tool"

    return "end"


# ============================================================
# 10. MiniGraph
# ============================================================

class MiniGraph:

    def __init__(self):

        self.nodes = {}

        self.edges = {}

        self.conditional_edges = {}

        self.start_node = None

    # ========================================================
    # Add Node
    # ========================================================

    def add_node(
        self,
        name,
        node,
    ):

        self.nodes[name] = node

    # ========================================================
    # Add Edge
    # ========================================================

    def add_edge(
        self,
        from_node,
        to_node,
    ):

        self.edges[from_node] = to_node

    # ========================================================
    # Add Conditional Edge
    # ========================================================

    def add_conditional_edges(
        self,
        node_name,
        router,
    ):

        self.conditional_edges[
            node_name
        ] = router

    # ========================================================
    # Set Entry Point
    # ========================================================

    def set_entry_point(
        self,
        node_name,
    ):

        self.start_node = node_name

    # ========================================================
    # Run
    # ========================================================

    def run(
        self,
        state: AgentState,
    ):

        current_node = (
            self.start_node
        )

        while current_node is not None:

            # ------------------------------------------------
            # Iteration
            # ------------------------------------------------

            state.iteration += 1

            print(
                f"\n--- Iteration "
                f"{state.iteration} ---"
            )

            # ------------------------------------------------
            # Safety limit
            # ------------------------------------------------

            if state.iteration > 10:

                raise RuntimeError(
                    "Graph exceeded maximum iterations."
                )

            # ------------------------------------------------
            # Get Node
            # ------------------------------------------------

            node = self.nodes[
                current_node
            ]

            # ------------------------------------------------
            # Execute Node
            #
            # State → State
            # ------------------------------------------------

            state = node(
                state
            )

            # ------------------------------------------------
            # Check Conditional Edge
            # ------------------------------------------------

            if (
                current_node
                in self.conditional_edges
            ):

                router = (
                    self.conditional_edges[
                        current_node
                    ]
                )

                next_node = router(
                    state
                )

                # --------------------------------------------
                # END
                # --------------------------------------------

                if next_node == "end":

                    break

                current_node = (
                    next_node
                )

                continue

            # ------------------------------------------------
            # Normal Edge
            # ------------------------------------------------

            if (
                current_node
                in self.edges
            ):

                current_node = (
                    self.edges[
                        current_node
                    ]
                )

                continue

            # ------------------------------------------------
            # No Edge
            # ------------------------------------------------

            break

        return state


# ============================================================
# 11. Build Graph
# ============================================================

def build_graph():

    graph = MiniGraph()

    # --------------------------------------------------------
    # Nodes
    # --------------------------------------------------------

    graph.add_node(
        "llm",
        llm_node,
    )

    graph.add_node(
        "tool",
        tool_node,
    )

    # --------------------------------------------------------
    # Entry
    # --------------------------------------------------------

    graph.set_entry_point(
        "llm"
    )

    # --------------------------------------------------------
    # Normal Edge
    #
    # tool → llm
    # --------------------------------------------------------

    graph.add_edge(
        "tool",
        "llm",
    )

    # --------------------------------------------------------
    # Conditional Edge
    #
    # llm → tool
    # llm → end
    # --------------------------------------------------------

    graph.add_conditional_edges(
        "llm",
        route_after_llm,
    )

    return graph


# ============================================================
# 12. Run Agent
# ============================================================

def run_agent(
    task: str,
):

    state = AgentState(
        task=task
    )

    graph = build_graph()

    final_state = graph.run(
        state
    )

    return final_state


# ============================================================
# 13. Main
# ============================================================

if __name__ == "__main__":

    task = (
        "请告诉我我的名字，然后计算 "
        "12345 + 6789，再把结果乘以 2。"
    )

    final_state = run_agent(
        task
    )

    # ========================================================
    # Final Answer
    # ========================================================

    print("\n" + "=" * 60)

    print(
        "FINAL ANSWER"
    )

    print(
        "=" * 60
    )

    print(
        final_state.final_answer
    )

    # ========================================================
    # Final State
    # ========================================================

    print("\n" + "=" * 60)

    print(
        "FINAL AGENT STATE"
    )

    print(
        "=" * 60
    )

    print(
        f"\nTask:\n"
        f"{final_state.task}"
    )

    print(
        f"\nIterations:\n"
        f"{final_state.iteration}"
    )

    print(
        f"\nFinished:\n"
        f"{final_state.finished}"
    )

    print(
        "\nTool Calls:"
    )

    for call in final_state.tool_calls:

        print(call)

    print(
        "\nTool Results:"
    )

    for result in final_state.tool_results:

        print(result)

## Trajectory 输出示例
```shell
============================================================
TRAJECTORY
============================================================

Step 1
Node: llm
Input: {'task': '请告诉我我的名字，然后计算 12345 + 6789，再把结果乘以2。', 'previous_tool_outputs': []}
Output: {'tool_calls': [{'name': 'get_user_name', 'arguments': '{}', 'call_id': 'call_PfJcZdN5PjLII5fnsP4iIH9w'}], 'final_answer': None}

Step 2
Node: tool
Input: {'tool': 'get_user_name', 'arguments': {}}
Output: {'result': 'Sylvie'}

Step 3
Node: llm
Input: {'task': '请告诉我我的名字，然后计算 12345 + 6789，再把结果乘以2。', 'previous_tool_outputs': [{'type': 'function_call_output', 'call_id': 'call_PfJcZdN5PjLII5fnsP4iIH9w', 'output': 'Sylvie'}]}
Output: {'tool_calls': [{'name': 'calculator', 'arguments': '{"a":12345,"b":6789,"operation":"add"}', 'call_id': 'call_ly3MyXQTVfJXNOjnptdK074a'}], 'final_answer': None}

Step 4
Node: tool
Input: {'tool': 'calculator', 'arguments': {'a': 12345, 'b': 6789, 'operation': 'add'}}
Output: {'result': 19134}

Step 5
Node: llm
Input: {'task': '请告诉我我的名字，然后计算 12345 + 6789，再把结果乘以2。', 'previous_tool_outputs': [{'type': 'function_call_output', 'call_id': 'call_ly3MyXQTVfJXNOjnptdK074a', 'output': '19134'}]}
Output: {'tool_calls': [{'name': 'calculator', 'arguments': '{"a":19134,"b":2,"operation":"multiply"}', 'call_id': 'call_q1oSKs2KUXpRum9XWIbmNkM8'}], 'final_answer': None}

Step 6
Node: tool
Input: {'tool': 'calculator', 'arguments': {'a': 19134, 'b': 2, 'operation': 'multiply'}}
Output: {'result': 38268}

Step 7
Node: llm
Input: {'task': '请告诉我我的名字，然后计算 12345 + 6789，再把结果乘以2。', 'previous_tool_outputs': [{'type': 'function_call_output', 'call_id': 'call_q1oSKs2KUXpRum9XWIbmNkM8', 'output': '38268'}]}
Output: {'tool_calls': [], 'final_answer': '你的名字是 Sylvie。\n计算结果：\n- 12345 + 6789 = 19134\n- 19134 × 2 = 38268'}
```
